In [ ]:
# Install dependencies
!pip install -q -r ../requirements.txt

import os
import sys
import torch

sys.path.append(os.path.abspath("../src"))

from finetune_crnn import TrainConfig, train_one_condition

print("Environment ready. CUDA:", torch.cuda.is_available())

## Optional Experiment: CRNN finetuning with augmentation

This notebook is designed as an **extra experiment** on top of the robustness study:

- We train a lightweight CRNN model with CTC loss on a **small training set** (`data/train`).
- You can compare two regimes by running the training twice:
  - **Without augmentation** (clean training images),
  - **With augmentation** (you can extend `OCRDataset` or the dataloader to apply the same transforms as in `augmentations.py`).
- Evaluation is done on the original (non‑augmented) `data/test` set using CER.

Because training even a small OCR model is heavy on CPU, this notebook **only runs when a CUDA GPU is available**. On a GPU, 10–20 epochs on a few hundred images is feasible within ~10–20 minutes.

In [ ]:
# Configure training and validation sets

train_dir = os.path.abspath("../data/train")
val_dir = os.path.abspath("../data/test")  # use test as a small validation set for the demo
out_dir = os.path.abspath("../results_crnn")

print("Train dir:", train_dir)
print("Val dir:", val_dir)
print("Out dir:", out_dir)

In [ ]:
# Skip if no GPU is available

if not torch.cuda.is_available():
    print("CUDA is not available; skipping finetuning experiment.")
else:
    cfg = TrainConfig(
        train_dir=train_dir,
        val_dir=val_dir,
        out_dir=out_dir,
        epochs=10,
        batch_size=8,
        lr=1e-3,
        seed=42,
    )
    best_cer = train_one_condition(cfg)
    print(f"Best validation CER: {best_cer:.3f}")